# SoundStage Separator: training on Kaggle

Run this notebook on Kaggle with a GPU (Settings: Accelerator **GPU T4 x2** or **P100**,
Internet **on**). Everything is downloaded on Kaggle's servers. See `training/README.md` for
the steps, the GPU-hour budget and the dataset licences.

Set `STAGE` below to the session you are running (1 to 4). Every stage resumes where it stopped,
so a session that times out is simply run again.

In [ ]:
TEACHER = "cascade:fast"  # "htdemucs" for htdemucs_ft alone, or cascade:best / cascade:fast
STAGE = 1          # 1 teacher stems, 2 and 3 training, 4 evaluation and upload
HOURS = 10.5       # training time per session; Kaggle stops GPU sessions at 12 hours
REPO = "danieltesh/soundstage-separator"

In [ ]:
import os
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")  # a write token, saved as a Kaggle secret
if not os.path.isdir("stage-spatial-audio"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/danielomoteshohacks/stage-spatial-audio.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "stage-spatial-audio",
                "demucs", "museval", "onnx", "onnxscript", "onnxruntime", "huggingface_hub"], check=True)

In [ ]:
# MUSDB18-HQ (Rafii et al. 2019, Zenodo 10.5281/zenodo.3338373): research use only.
# Use an attached Kaggle copy if you added one, otherwise download it here (about 30 GB).
import glob
import pathlib
import urllib.request
import zipfile

found = glob.glob("/kaggle/input/**/musdb18hq*/train", recursive=True) or glob.glob("/kaggle/input/**/train/*/vocals.wav", recursive=True)
if found:
    MUSDB = str(pathlib.Path(found[0]) if found[0].endswith("train") else pathlib.Path(found[0]).parents[1])
    MUSDB = MUSDB[:-len("/train")] if MUSDB.endswith("/train") else MUSDB
else:
    MUSDB = "/kaggle/working/musdb18hq"
    if not os.path.isdir(MUSDB + "/test"):
        urllib.request.urlretrieve("https://zenodo.org/records/3338373/files/musdb18hq.zip?download=1", "/kaggle/working/musdb18hq.zip")
        zipfile.ZipFile("/kaggle/working/musdb18hq.zip").extractall(MUSDB)
        os.remove("/kaggle/working/musdb18hq.zip")
print("MUSDB18-HQ at", MUSDB, os.listdir(MUSDB))

In [ ]:
# Bring back earlier sessions' work from the model repository (checkpoint and teacher stems are
# kept in /kaggle/working between runs only if you save a version; the repository always has them).
from huggingface_hub import snapshot_download

os.makedirs("/kaggle/working/ssep", exist_ok=True)
try:
    snapshot_download(REPO, local_dir="/kaggle/working/ssep", allow_patterns=["ckpt.pt", "train_log.jsonl"], token=os.environ["HF_TOKEN"])
except Exception as exc:
    print("no earlier checkpoint:", exc)

In [ ]:
if STAGE == 1:
    # Teacher pseudo-labels for distillation: the v3 separation cascade (decision 0023) over the
    # training set. Set TEACHER = "htdemucs" to use htdemucs_ft alone (about 1.5 h on a T4); the
    # cascade "fast" preset runs two neural levels and takes longer, so run it on one T4 first on
    # a few tracks (--limit is not needed: finished tracks are skipped on a re-run).
    subprocess.run([sys.executable, "stage-spatial-audio/training/make_teacher_stems.py",
                    "--input", MUSDB + "/train", "--out", "/kaggle/working/teacher/musdb",
                    "--teacher", TEACHER, "--models", "htdemucs_ft"], check=True)
    # Extra music you have the right to use: put a Kaggle dataset of audio files at EXTRA.
    EXTRA = "/kaggle/input/soundstage-extra-music"
    if os.path.isdir(EXTRA):
        subprocess.run([sys.executable, "stage-spatial-audio/training/make_teacher_stems.py",
                        "--input", EXTRA, "--out", "/kaggle/working/teacher/extra",
                        "--teacher", TEACHER, "--models", "htdemucs_ft"], check=True)

In [ ]:
if STAGE in (2, 3):
    # Teacher stems from the STAGE 1 run: attach that run's output (Add Input > Your Work) and
    # it appears under /kaggle/input; or they are still in /kaggle/working in the same session.
    def find(sub):
        hits = glob.glob(f"/kaggle/input/**/{sub}", recursive=True) + glob.glob(f"/kaggle/working/{sub}")
        return next((h for h in hits if os.path.isdir(h)), None)
    teacher, extra = find("teacher/musdb"), find("teacher/extra")
    cmd = [sys.executable, "stage-spatial-audio/training/train.py", "--musdb", MUSDB + "/train",
           "--out", "/kaggle/working/ssep", "--hours", str(HOURS), "--push", REPO]
    if teacher:
        cmd += ["--teacher", teacher]
    if extra:
        cmd += ["--extra", extra]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
if STAGE == 4:
    # MUSDB18-HQ test set: ours, the teacher and htdemucs, with size and time; writes the result
    # into model.json, which decides whether the Space may use our engine by default.
    from huggingface_hub import HfApi, hf_hub_download
    for f in ("model.onnx", "model.json"):
        hf_hub_download(REPO, f, local_dir="/kaggle/working/ssep", token=os.environ["HF_TOKEN"])
    subprocess.run([sys.executable, "stage-spatial-audio/scripts/eval_separation.py",
                    "--musdb", MUSDB + "/test", "--engines", "ssep,htdemucs,htdemucs_ft",
                    "--ssep", "/kaggle/working/ssep", "--out", "/kaggle/working/eval_separation.json",
                    "--update-card", "/kaggle/working/ssep/model.json"], check=True)
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.upload_file(path_or_fileobj="/kaggle/working/ssep/model.json", path_in_repo="model.json", repo_id=REPO)
    api.upload_file(path_or_fileobj="/kaggle/working/eval_separation.json", path_in_repo="eval_separation.json", repo_id=REPO)
    print(open("/kaggle/working/eval_separation.json").read())